# Earthquake Magnitude v5 — Training

P-wave-anchored CNN-BiLSTM regression model. Predicts earthquake magnitude from a 30-second waveform window anchored at the P-wave arrival. Run `data_collection_v5.ipynb` first.

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIG                                            ║
# ╚══════════════════════════════════════════════════════════════╝

PROCESSED_DIR        = 'processed_v5'
MODEL_DIR            = 'models_v5'
RESULTS_DIR          = 'results_v5'

# ── Sample rate ──────────────────────────────────────────────────────────────
TARGET_SR            = 20
WINDOW_SEC           = 30.0
PRE_P_SEC            = 1.0
WINDOW_SIZE          = int(WINDOW_SEC * TARGET_SR)   # 600 samples
PRE_P_SAMPLES        = int(PRE_P_SEC  * TARGET_SR)   # 20  samples
N_LOC                = 4

# ── Training hyperparameters ─────────────────────────────────────────────────
LEARNING_RATE        = 0.0005
EPOCHS               = 100
BATCH_SIZE           = 64
EARLY_STOP_PATIENCE  = 15
REDUCE_LR_PATIENCE   = 7

# ── Sample weights — dynamic inverse-frequency balancing ─────────────────────
# Weights are computed automatically from the training set distribution:
#   weight_class = total_samples / (n_classes × samples_in_class)
# This means every magnitude class contributes equally to the total loss
# regardless of how many samples it has. A class with 100× fewer samples
# gets 100× the weight — perfectly balanced.
USE_SAMPLE_WEIGHTS   = False   # best run had no weights
WEIGHT_SCHEME        = 'inverse_frequency'   # computed in Cell 4b from actual data
MAX_WEIGHT           = 10.0                  # cap — prevents extreme classes from destabilising training

# Magnitude bins (USGS seismological standard)
MAG_BINS   = [0.0, 4.5, 5.5, 6.5, 99.0]
BIN_NAMES  = ['Minor', 'Moderate', 'Strong', 'Major']

# ── Split ────────────────────────────────────────────────────────────────────
SPLIT_MODE           = 'chronological'
TRAIN_FRACTION       = 0.80
VAL_FRACTION         = 0.10

MAG_MIN              = 2.5
MAG_MAX              = 8.5

import os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

os.makedirs(MODEL_DIR,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Config loaded.')
print(f'  TARGET_SR          : {TARGET_SR} Hz  ({WINDOW_SIZE} samples per window)')
print(f'  EPOCHS             : {EPOCHS}  (patience={EARLY_STOP_PATIENCE})')
print(f'  SPLIT_MODE         : {SPLIT_MODE}')
print(f'  USE_SAMPLE_WEIGHTS : {USE_SAMPLE_WEIGHTS}  ({WEIGHT_SCHEME})')
print(f'  MAG_BINS           : {MAG_BINS}  →  {BIN_NAMES}')

Config loaded.
  TARGET_SR          : 20 Hz  (600 samples per window)
  EPOCHS             : 100  (patience=15)
  SPLIT_MODE         : chronological
  USE_SAMPLE_WEIGHTS : True  (inverse_frequency)
  MAG_BINS           : [0.0, 4.5, 5.5, 6.5, 99.0]  →  ['Minor', 'Moderate', 'Strong', 'Major']


## Cell 2 — GPU Check

Verifies that TensorFlow can see the Apple Metal GPU. Enables memory growth so TF allocates memory gradually instead of reserving everything at start-up (prevents the 'Dst tensor not initialized' crash on 8 GB unified memory).

In [2]:
# CELL 2 — GPU CHECK + MEMORY GROWTH

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'Metal GPU active  : {gpus[0].name}')
    print('Memory growth     : enabled (allocates as needed)')
else:
    print('No GPU found — training will use CPU.')

print(f'TensorFlow {tf.__version__}')


Metal GPU active  : /physical_device:GPU:0
Memory growth     : enabled (allocates as needed)
TensorFlow 2.16.2


## Cell 3 — Imports

Loads all Python libraries needed for data processing, model building, training, and evaluation.

In [3]:
# CELL 3 — IMPORTS

import math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout,
                                      BatchNormalization, Bidirectional,
                                      LSTM, Dense, Concatenate,
                                      MultiHeadAttention, LayerNormalization,
                                      GlobalAveragePooling1D, Add)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

AUTOTUNE = tf.data.AUTOTUNE
print('Imports OK.')


Imports OK.


## Cell 4 — Data Loading

For every CSV event file:
1. Reads the waveform amplitude and phase labels.
2. Finds the first P-wave sample (label == 1).
3. Cuts a 30-second window: 1s before P-arrival to 29s after.
4. Z-score normalises the window.
5. Looks up the earthquake magnitude from the matching `_meta.json`.

Returns three arrays: waveform windows, location features, magnitude targets.

In [4]:
# CELL 4 — DATA LOADING + SPLIT
# • Resamples every waveform to TARGET_SR Hz (fixes 20/40 Hz station data)
# • SPLIT_MODE='chronological' — pools ALL events, sorts by origin_time,
#   then splits: first 80% train, next 10% val, last 10% test (no leakage)
# • SPLIT_MODE='time' — uses the original filename label (event_train_* / event_test_*)
# • SPLIT_MODE='random' — shuffled 80/10/10 split (not recommended for time-series)

from math import gcd
from scipy.signal import resample_poly

WINDOW_SAMP = int(WINDOW_SEC * TARGET_SR)   # 600 @ 20 Hz
PRE_P_SAMP  = int(PRE_P_SEC  * TARGET_SR)   # 20  @ 20 Hz


def find_meta(csv_path):
    """Find the _meta.json that belongs to a CSV event file."""
    p = Path(csv_path)
    # Same directory (processed_v5/ keeps meta alongside csv)
    meta_same = p.parent / f'{p.stem}_meta.json'
    if meta_same.exists():
        return meta_same
    # Fallback: waveform dirs
    for wave_dir in ['waveforms_v5', 'waveforms_temporal', '../waveforms_temporal']:
        meta_wave = Path(wave_dir) / f'{p.stem}_meta.json'
        if meta_wave.exists():
            return meta_wave
    return None


def _resample(amp, label, sr_orig, sr_target):
    """Poly-phase resample amplitude + label array from sr_orig → sr_target Hz."""
    if abs(sr_orig - sr_target) < 0.5:
        return amp, label
    sr_orig, sr_target = int(round(sr_orig)), int(sr_target)
    g        = gcd(sr_orig, sr_target)
    up, down = sr_target // g, sr_orig // g
    amp_r    = resample_poly(amp, up, down).astype(np.float32)
    idx      = np.round(np.linspace(0, len(label) - 1, len(amp_r))).astype(int)
    return amp_r, label[idx]


def _load_all_events():
    """
    Load every usable event CSV from PROCESSED_DIR.
    Returns arrays plus origin_times (for chronological sort) and
    split_labels (for time-based split fallback).
    """
    files   = sorted(Path(PROCESSED_DIR).glob('*.csv'))
    X_wave, X_loc, y_mag = [], [], []
    origin_times, split_labels = [], []

    skipped = {'no_meta': 0, 'bad_meta': 0, 'mag_range': 0,
               'no_p_wave': 0, 'too_short': 0, 'bad_csv': 0, 'bad_sr': 0}

    for fp in files:
        # ── read CSV ──────────────────────────────────────────────
        try:
            df = pd.read_csv(fp)
        except Exception:
            skipped['bad_csv'] += 1
            continue

        file_split = 'train' if '_train_' in fp.name else 'test'

        # ── infer original sample rate ────────────────────────────
        try:
            dt      = float(df['time_sec'].iloc[1]) - float(df['time_sec'].iloc[0])
            sr_orig = 1.0 / dt
            if sr_orig < 5 or sr_orig > 1000:
                skipped['bad_sr'] += 1
                continue
        except Exception:
            skipped['bad_sr'] += 1
            continue

        # ── meta: magnitude + origin_time ─────────────────────────
        meta_path = find_meta(str(fp))
        if meta_path is None:
            skipped['no_meta'] += 1
            continue
        try:
            meta = json.load(open(str(meta_path)))
            mag  = float(meta['magnitude'])
            # Parse origin_time — stored as ISO string e.g. "2020-02-01 09:43:55+00:00"
            ot_raw = meta.get('origin_time', '')
            ot     = pd.Timestamp(ot_raw)     # robust parser handles +00:00 suffix
        except Exception:
            skipped['bad_meta'] += 1
            continue

        if not (MAG_MIN <= mag <= MAG_MAX):
            skipped['mag_range'] += 1
            continue

        # ── resample to TARGET_SR ─────────────────────────────────
        amp    = df['amplitude'].values.astype(np.float32)
        labels = df['label'].fillna(0).round().astype(int).values
        amp, labels = _resample(amp, labels, sr_orig, TARGET_SR)

        # ── find P-wave arrival ───────────────────────────────────
        p_idx_arr = np.where(labels == 1)[0]
        if len(p_idx_arr) == 0:
            skipped['no_p_wave'] += 1
            continue
        p_idx = int(p_idx_arr[0])

        # ── cut 30-second window anchored 1s before P-wave ────────
        start = max(0, p_idx - PRE_P_SAMP)
        end   = start + WINDOW_SAMP
        if end > len(amp):
            skipped['too_short'] += 1
            continue

        # ── z-score normalise ─────────────────────────────────────
        w = amp[start:end].reshape(WINDOW_SAMP, 1)
        w = (w - w.mean()) / (w.std() + 1e-8)

        # ── location features ─────────────────────────────────────
        mid = min((start + end) // 2, len(df) - 1)
        loc = [float(df['dist_km_n'].iloc[mid]),
               float(df['depth_km_n'].iloc[mid]),
               float(df['lat_n'].iloc[mid]),
               float(df['lon_n'].iloc[mid])]

        X_wave.append(w)
        X_loc.append(loc)
        y_mag.append(mag)
        origin_times.append(ot)
        split_labels.append(file_split)

    total_skip = sum(skipped.values())
    if total_skip:
        print(f'  Skipped {total_skip}: {skipped}')
    print(f'  Total usable events : {len(y_mag):,}  (all resampled to {TARGET_SR} Hz)')
    return (np.array(X_wave, dtype=np.float32),
            np.array(X_loc,  dtype=np.float32),
            np.array(y_mag,  dtype=np.float32),
            origin_times,
            split_labels)


# ── Load all events ───────────────────────────────────────────────────────
print('Loading all events from', PROCESSED_DIR, '...')
Xw_all, Xl_all, y_all, origin_times, split_labels = _load_all_events()
n_total = len(y_all)

# ── Apply split ───────────────────────────────────────────────────────────
if SPLIT_MODE == 'chronological':
    # Sort every event by its earthquake origin time (oldest → newest)
    sort_idx = np.argsort([ot.value for ot in origin_times])   # .value = int64 ns
    Xw_all   = Xw_all[sort_idx]
    Xl_all   = Xl_all[sort_idx]
    y_all    = y_all[sort_idx]

    n_tr  = int(n_total * TRAIN_FRACTION)           # first 80%
    n_val = int(n_total * VAL_FRACTION)             # next  10%
    # remaining                                     # last  10%

    Xw_train, Xl_train, y_train = Xw_all[:n_tr],            Xl_all[:n_tr],            y_all[:n_tr]
    Xw_val,   Xl_val,   y_val   = Xw_all[n_tr:n_tr+n_val],  Xl_all[n_tr:n_tr+n_val],  y_all[n_tr:n_tr+n_val]
    Xw_test,  Xl_test,  y_test  = Xw_all[n_tr+n_val:],      Xl_all[n_tr+n_val:],      y_all[n_tr+n_val:]

    # Report the actual time ranges for each split
    ot_sorted = [origin_times[i] for i in sort_idx]
    tr_range  = (ot_sorted[0].date(),             ot_sorted[n_tr - 1].date())
    val_range = (ot_sorted[n_tr].date(),           ot_sorted[n_tr + n_val - 1].date())
    te_range  = (ot_sorted[n_tr + n_val].date(),   ot_sorted[-1].date())

    print(f'  Split: chronological  (oldest → newest, no data leakage)')
    print(f'    train : {tr_range[0]}  →  {tr_range[1]}')
    print(f'    val   : {val_range[0]}  →  {val_range[1]}')
    print(f'    test  : {te_range[0]}  →  {te_range[1]}')

elif SPLIT_MODE == 'random':
    rng   = np.random.default_rng(42)
    idx   = rng.permutation(n_total)
    n_tr  = int(n_total * TRAIN_FRACTION)
    n_val = int(n_total * VAL_FRACTION)

    Xw_train, Xl_train, y_train = Xw_all[idx[:n_tr]],            Xl_all[idx[:n_tr]],            y_all[idx[:n_tr]]
    Xw_val,   Xl_val,   y_val   = Xw_all[idx[n_tr:n_tr+n_val]],  Xl_all[idx[n_tr:n_tr+n_val]],  y_all[idx[n_tr:n_tr+n_val]]
    Xw_test,  Xl_test,  y_test  = Xw_all[idx[n_tr+n_val:]],      Xl_all[idx[n_tr+n_val:]],      y_all[idx[n_tr+n_val:]]
    print(f'  Split: random (seed=42)')

else:   # 'time' — keep original filename labels
    sl        = np.array(split_labels)
    tr_mask   = sl == 'train'
    te_mask   = sl == 'test'
    Xw_tr_all = Xw_all[tr_mask]; Xl_tr_all = Xl_all[tr_mask]; y_tr_all = y_all[tr_mask]
    n_val     = max(1, int(len(y_tr_all) * VAL_FRACTION))
    Xw_train, Xl_train, y_train = Xw_tr_all[:-n_val], Xl_tr_all[:-n_val], y_tr_all[:-n_val]
    Xw_val,   Xl_val,   y_val   = Xw_tr_all[-n_val:], Xl_tr_all[-n_val:], y_tr_all[-n_val:]
    Xw_test,  Xl_test,  y_test  = Xw_all[te_mask],    Xl_all[te_mask],    y_all[te_mask]
    print(f'  Split: time-based (train=event_train_* / test=event_test_*)')

print()
print('=' * 55)
print(f'  Train  : {len(y_train):>6,}  mag {y_train.min():.2f}–{y_train.max():.2f}')
print(f'  Val    : {len(y_val):>6,}  mag {y_val.min():.2f}–{y_val.max():.2f}')
print(f'  Test   : {len(y_test):>6,}  mag {y_test.min():.2f}–{y_test.max():.2f}')
print(f'  Total  : {n_total:>6,}')
print('=' * 55)

if len(y_train) < 200:
    print(f'\nWARNING: Only {len(y_train)} train events — model will overfit. Need 1,000+ for good results.')

Loading all events from processed_v5 ...


  Skipped 300: {'no_meta': 0, 'bad_meta': 0, 'mag_range': 0, 'no_p_wave': 0, 'too_short': 300, 'bad_csv': 0, 'bad_sr': 0}
  Total usable events : 29,037  (all resampled to 20 Hz)
  Split: chronological  (oldest → newest, no data leakage)
    train : 2000-02-02  →  2011-03-14
    val   : 2011-03-14  →  2020-10-05
    test  : 2020-10-06  →  2022-12-31

  Train  : 23,229  mag 3.00–8.16
  Val    :  2,903  mag 3.10–7.10
  Test   :  2,905  mag 3.50–7.10
  Total  : 29,037


## Cell 5 — tf.data Pipeline

Wraps the NumPy arrays in a `tf.data.Dataset` so data is streamed to the GPU in batches. The train dataset is shuffled each epoch. `prefetch` overlaps data preparation with GPU compute.

In [5]:
# CELL 4b — DYNAMIC INVERSE-FREQUENCY SAMPLE WEIGHTS (CAPPED)
# ─────────────────────────────────────────────────────────────────────────────
# Formula:  weight_class = n_total / (n_classes × count_in_class)
# Cap:      weights are clipped to MAX_WEIGHT (set in Cell 1, default 10×)
#
# Why cap? With only 3 Major events in the test set, uncapped weights reached
# ~82×. The model destabilised — it kept overcorrecting for those 3 examples,
# hurt MAE on the other 2,902, and stopped at epoch 18 with R²=−0.55.
# Capping at 10× keeps Major events influential without dominating the loss.
# ─────────────────────────────────────────────────────────────────────────────

def _bin_index(mag_arr):
    """Convert magnitude array → bin index (0=Minor,1=Moderate,2=Strong,3=Major)."""
    return np.digitize(mag_arr, MAG_BINS[1:-1]).astype(int)

def _compute_capped_weights(y_mag, max_weight=MAX_WEIGHT):
    """Inverse-frequency weights capped at max_weight to prevent training instability."""
    n_total   = len(y_mag)
    n_classes = len(BIN_NAMES)
    bins      = _bin_index(y_mag)
    w         = np.ones(n_total, dtype=np.float32)
    print(f'  Inverse-frequency weights (capped at {max_weight}×):')
    print('  ' + '─' * 60)
    for i, name in enumerate(BIN_NAMES):
        count = int((bins == i).sum())
        if count == 0:
            weight = 1.0
        else:
            weight = min(n_total / (n_classes * count), max_weight)
        w[bins == i] = weight
        bar = '█' * min(int(count / n_total * 50), 50)
        flag = '  ← CAPPED' if (count > 0 and n_total / (n_classes * count) > max_weight) else ''
        print(f'  {name:<10} count={count:>6,}  weight={weight:>6.2f}×  {bar}{flag}')
    print('  ' + '─' * 60)
    print(f'  Mean weight : {w.mean():.3f}')
    print(f'  Max weight  : {w.max():.2f}× (cap={max_weight}×)')
    return w

print('Computing capped inverse-frequency weights ...')
sw_train = _compute_capped_weights(y_train, max_weight=MAX_WEIGHT)

# Val uses uniform weights (evaluate on real distribution)
sw_val = np.ones(len(y_val), dtype=np.float32)

print(f'\n  Test set class breakdown (no weights — real distribution):')
test_bins = _bin_index(y_test)
for i, name in enumerate(BIN_NAMES):
    print(f'    {name:<10} {int((test_bins==i).sum()):>5,} events')


Computing capped inverse-frequency weights ...
  Inverse-frequency weights (capped at 10.0×):
  ────────────────────────────────────────────────────────────
  Minor      count=13,328  weight=  0.44×  ████████████████████████████
  Moderate   count= 9,076  weight=  0.64×  ███████████████████
  Strong     count=   749  weight=  7.75×  █
  Major      count=    76  weight= 10.00×    ← CAPPED
  ────────────────────────────────────────────────────────────
  Mean weight : 0.783
  Max weight  : 10.00× (cap=10.0×)

  Test set class breakdown (no weights — real distribution):
    Minor      1,658 events
    Moderate   1,193 events
    Strong        51 events
    Major          3 events


In [6]:
# CELL 5 — tf.data PIPELINE

def make_dataset(Xw, Xl, yy, sample_weights=None, shuffle=False):
    """Wrap arrays in a tf.data.Dataset.

    If sample_weights is provided, the dataset yields
    (inputs, targets, weights) triples — Keras uses the weights
    to scale the per-sample loss during training.
    """
    if sample_weights is not None:
        ds = tf.data.Dataset.from_tensor_slices(
            ({'waveform': Xw, 'location': Xl}, yy,
             sample_weights.astype(np.float32)))
    else:
        ds = tf.data.Dataset.from_tensor_slices(
            ({'waveform': Xw, 'location': Xl}, yy))

    if shuffle:
        ds = ds.shuffle(buffer_size=min(10000, len(yy)), seed=42)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(Xw_train, Xl_train, y_train,
                        sample_weights=(sw_train if USE_SAMPLE_WEIGHTS else None),
                        shuffle=True)
val_ds   = make_dataset(Xw_val,   Xl_val,   y_val,
                        sample_weights=(sw_val   if USE_SAMPLE_WEIGHTS else None))
test_ds  = make_dataset(Xw_test,  Xl_test,  y_test)   # no weights at eval time

print('Datasets ready.')
print(f'  train batches : {len(train_ds)}')
print(f'  val   batches : {len(val_ds)}')
print(f'  test  batches : {len(test_ds)}')
print(f'  sample weights: {"ON — Strong×10, Major×20" if USE_SAMPLE_WEIGHTS else "OFF"}')

Datasets ready.
  train batches : 363
  val   batches : 46
  test  batches : 46
  sample weights: ON — Strong×10, Major×20


## Cell 6 — Model Architecture

Dual-input CNN-BiLSTM regression model:
- **Waveform branch**: three Conv1D blocks extract features, then two BiLSTM layers capture temporal patterns.
- **Location branch**: two Dense layers encode station distance, depth, lat, lon.
- **Fusion**: concatenated features feed into a final Dense head that outputs a single magnitude value.

In [7]:
# CELL 6 — MODEL ARCHITECTURE (Deeper CNN + BiLSTM + Multi-Head Attention)
# v5.1 — Added 4th Conv1D block (256 filters) with residual connection
# Total params: ~750k (up from ~400k)

# ── Waveform branch ────────────────────────────────────────────
wave_in = Input(shape=(WINDOW_SIZE, 1), name='waveform')

# Block 1
x = Conv1D(32,  kernel_size=7, padding='same', activation='relu')(wave_in)
x = BatchNormalization()(x)
x = MaxPooling1D(2)(x)
x = Dropout(0.2)(x)

# Block 2
x = Conv1D(64,  kernel_size=5, padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling1D(2)(x)
x = Dropout(0.2)(x)

# Block 3
x = Conv1D(128, kernel_size=3, padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling1D(2)(x)
x = Dropout(0.2)(x)

# Block 4 — NEW: deeper features, residual skip connection
residual = Conv1D(256, kernel_size=1, padding='same')(x)   # 1×1 conv to match dims
x = Conv1D(256, kernel_size=3, padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = Conv1D(256, kernel_size=3, padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = Add()([x, residual])                                   # residual skip
x = Dropout(0.2)(x)

# BiLSTM — keeps sequences so attention can look across all time steps
x = Bidirectional(LSTM(64, return_sequences=True))(x)

# ── Multi-Head Attention ───────────────────────────────────────
attn_out = MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.1)(
    query=x, value=x, key=x)
x = Add()([x, attn_out])
x = LayerNormalization()(x)

# Second BiLSTM collapses the sequence to a single vector
x = Bidirectional(LSTM(32))(x)

x = Dense(64, activation='relu')(x)
wave_out = Dropout(0.3)(x)

# ── Location branch ────────────────────────────────────────────
loc_in  = Input(shape=(N_LOC,), name='location')
lx      = Dense(32, activation='relu')(loc_in)
lx      = BatchNormalization()(lx)
loc_out = Dense(16, activation='relu')(lx)

# ── Fusion ─────────────────────────────────────────────────────
merged  = Concatenate()([wave_out, loc_out])
fused   = Dense(64, activation='relu')(merged)
fused   = Dropout(0.3)(fused)
out     = Dense(1, activation='linear', name='magnitude')(fused)

model = Model(inputs=[wave_in, loc_in], outputs=out)
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='mse', metrics=['mae'])

model.summary()
total_params = model.count_params()
print(f'\nTotal parameters : {total_params:,}')
print(f'Input shape      : ({WINDOW_SIZE}, 1)  —  {WINDOW_SEC}s at {TARGET_SR} Hz')
print(f'Architecture     : 4x Conv1D (32→64→128→256) + ResBlock + BiLSTM×2 + MHA')


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ waveform            │ (None, 600, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 600, 32)   │        256 │ waveform[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 600, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 300, 32)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 300, 32)   │          0 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 300, 64)   │     10,304 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 300, 64)   │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 150, 64)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 150, 64)   │          0 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 150, 128)  │     24,704 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 150, 128)  │        512 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 75, 128)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 75, 128)   │          0 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 75, 128)   │     98,816 │ dropout_2[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 75, 128)   │     66,048 │ bidirectional[0]… │
│ (MultiHeadAttentio… │                   │            │ bidirectional[0]… │
│                     │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 75, 128)   │          0 │ bidirectional[0]… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 75, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ location            │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 252,721 (987.19 KB)

 Trainable params: 252,209 (985.19 KB)

 Non-trainable params: 512 (2.00 KB)


Total parameters : 252,721
Input shape      : (600, 1)  —  30.0s at 20 Hz


## Cell 7 — Train

Trains the model for up to 50 epochs. `EarlyStopping` halts training when validation MAE stops improving. `ReduceLROnPlateau` halves the learning rate when the loss plateaus. The best weights are restored automatically.

In [8]:
# CELL 7 — TRAIN

from datetime import datetime
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

best_model_path = f'{MODEL_DIR}/best_model_v5_{TIMESTAMP}.keras'

callbacks = [
    EarlyStopping(
        monitor='val_mae',
        patience=EARLY_STOP_PATIENCE,
        restore_best_weights=True,
        verbose=1),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=REDUCE_LR_PATIENCE,
        min_lr=1e-6,
        verbose=1),
    ModelCheckpoint(
        filepath=best_model_path,
        monitor='val_mae',
        save_best_only=True,
        verbose=0),
]

print(f'Train: {len(y_train):,}  Val: {len(y_val):,}  '
      f'Batch: {BATCH_SIZE}  Max epochs: {EPOCHS}')
print()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1)

epochs_ran = len(history.history['loss'])
best_val_mae = min(history.history['val_mae'])
print()
print(f'Training complete.')
print(f'  Epochs ran    : {epochs_ran}')
print(f'  Best val MAE  : {best_val_mae:.4f}')


Train: 23,229  Val: 2,903  Batch: 64  Max epochs: 100

Epoch 1/100


  1/363 ━━━━━━━━━━━━━━━━━━━━ 27:22 5s/step - loss: 26.3783 - mae: 4.7202

  2/363 ━━━━━━━━━━━━━━━━━━━━ 2:07 354ms/step - loss: 26.1448 - mae: 4.6210

  3/363 ━━━━━━━━━━━━━━━━━━━━ 1:36 268ms/step - loss: 25.4960 - mae: 4.5383

  4/363 ━━━━━━━━━━━━━━━━━━━━ 1:24 236ms/step - loss: 24.9442 - mae: 4.4668

...

363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step - loss: 1.3369 - mae: 0.9079

363/363 ━━━━━━━━━━━━━━━━━━━━ 66s 183ms/step - loss: 1.1675 - mae: 0.8507 - val_loss: 0.2813 - val_mae: 0.4343 - learning_rate: 2.5000e-04


Epoch 17: early stopping


Restoring model weights from the end of the best epoch: 2.



Training complete.
  Epochs ran    : 17
  Best val MAE  : 0.3427


## Cell 8 — Evaluate

Runs the trained model on the held-out test set (2020–2023 events it has never seen). Reports MAE, RMSE, and R². Generates three diagnostic plots.

In [9]:
# CELL 8 — EVALUATE ON TEST SET

y_pred_raw = model.predict(test_ds, verbose=0)
y_pred     = y_pred_raw.flatten()
y_true     = y_test

mae  = mean_absolute_error(y_true, y_pred)
rmse = math.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)

print()
print('══════════════════════════════════════════')
print('  TEST RESULTS — Earthquake Magnitude v5  ')
print('══════════════════════════════════════════')
print(f'  MAE  : {mae:.4f} magnitude units')
print(f'  RMSE : {rmse:.4f}')
print(f'  R²   : {r2:.4f}')
print('══════════════════════════════════════════')

# ── Plot 1: Training curves ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['mae'],     label='Train MAE')
axes[0].plot(history.history['val_mae'], label='Val MAE')
axes[0].set_title('MAE per Epoch')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MAE (magnitude units)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train Loss (MSE)')
axes[1].plot(history.history['val_loss'], label='Val Loss (MSE)')
axes[1].set_title('Loss per Epoch')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves_v5_{TIMESTAMP}.png', dpi=150)
plt.close()
print('Saved: training_curves')

# ── Plot 2: Predicted vs True magnitude ───────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_true, y_pred, alpha=0.3, s=10, label='Events')
lims = [max(MAG_MIN, y_true.min() - 0.2), min(MAG_MAX, y_true.max() + 0.2)]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('True Magnitude'); ax.set_ylabel('Predicted Magnitude')
ax.set_title('Predicted vs True Magnitude')
ax.annotate(f'MAE={mae:.3f}\nR²={r2:.3f}', xy=(0.05, 0.88),
            xycoords='axes fraction', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5))
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/pred_vs_true_v5_{TIMESTAMP}.png', dpi=150)
plt.close()
print('Saved: pred_vs_true')

# ── Plot 3: Error distribution ────────────────────────────────
errors = y_pred - y_true
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(errors, bins=60, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(0, color='red', linewidth=1.5, linestyle='--', label='Perfect prediction')
ax.axvline(errors.mean(), color='orange', linewidth=1.5, linestyle='-',
           label=f'Mean error={errors.mean():.3f}')
ax.set_xlabel('Prediction Error (predicted − true magnitude)')
ax.set_ylabel('Count')
ax.set_title('Error Distribution')
ax.annotate(f'Mean={errors.mean():.3f}\nStd={errors.std():.3f}',
            xy=(0.78, 0.88), xycoords='axes fraction', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5))
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/error_dist_v5_{TIMESTAMP}.png', dpi=150)
plt.close()
print('Saved: error_dist')



══════════════════════════════════════════
  TEST RESULTS — Earthquake Magnitude v5  
══════════════════════════════════════════
  MAE  : 0.3195 magnitude units
  RMSE : 0.4077
  R²   : -0.2881
══════════════════════════════════════════


Saved: training_curves
Saved: pred_vs_true
Saved: error_dist


## Cell 8b — Confusion Matrix & Classification Accuracy

Magnitude prediction is a regression task, but we can bin predictions into seismological categories to measure classification accuracy. Also computes tolerance accuracy (±0.3 and ±0.5 magnitude units) — the standard used in published seismology papers.

In [10]:
# CELL 8b — CONFUSION MATRIX + FULL VALIDATION SUITE
# ─────────────────────────────────────────────────────────────────────────────
# Magnitude bins (USGS / seismology standard):
#   Minor     M < 4.5
#   Moderate  4.5 ≤ M < 5.5
#   Strong    5.5 ≤ M < 6.5
#   Major     6.5 ≤ M
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.metrics import confusion_matrix, classification_report
import itertools

# ── 1. Magnitude → class label ────────────────────────────────────────────────
BIN_EDGES  = [0, 4.5, 5.5, 6.5, 99]
BIN_LABELS = ['Minor\n(<4.5)', 'Moderate\n(4.5–5.5)', 'Strong\n(5.5–6.5)', 'Major\n(≥6.5)']
BIN_SHORT  = ['Minor', 'Moderate', 'Strong', 'Major']

def mag_to_class(mag_arr):
    return np.digitize(mag_arr, BIN_EDGES[1:-1])   # 0,1,2,3

y_true_cls = mag_to_class(y_true)
y_pred_cls = mag_to_class(y_pred)

present_classes = sorted(set(y_true_cls) | set(y_pred_cls))
present_labels  = [BIN_LABELS[i] for i in present_classes]
present_short   = [BIN_SHORT[i]  for i in present_classes]

# ── 2. Classification accuracy ────────────────────────────────────────────────
class_acc   = (y_true_cls == y_pred_cls).mean()
within_half = (np.abs(y_pred - y_true) <= 0.5).mean()
within_03   = (np.abs(y_pred - y_true) <= 0.3).mean()
within_01   = (np.abs(y_pred - y_true) <= 0.1).mean()

print('══════════════════════════════════════════════════════')
print('  FULL VALIDATION SUITE — earthd v5                  ')
print('══════════════════════════════════════════════════════')
print()
print('  Regression Metrics')
print(f'    MAE          : {mae:.4f}  mag units')
print(f'    RMSE         : {rmse:.4f}  mag units')
print(f'    R²           : {r2:.4f}')
print(f'    Bias (mean)  : {(y_pred-y_true).mean():.4f}  mag units')
print(f'    Std of error : {(y_pred-y_true).std():.4f}  mag units')
print()
print('  Tolerance Accuracy (% predictions within ±N mag units)')
print(f'    ±0.5  mag  :  {within_half*100:.1f}%   (publishable standard)')
print(f'    ±0.3  mag  :  {within_03*100:.1f}%   (good)')
print(f'    ±0.1  mag  :  {within_01*100:.1f}%   (excellent)')
print()
print('  Classification Accuracy (binned into 4 magnitude classes)')
print(f'    Class acc  :  {class_acc*100:.1f}%')
print()
print('  Classification Report')
print(classification_report(y_true_cls, y_pred_cls,
                            labels=present_classes,
                            target_names=present_short,
                            zero_division=0))

# ── 3. Confusion matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_true_cls, y_pred_cls, labels=present_classes)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, data, title, fmt in [
    (axes[0], cm,      'Confusion Matrix (counts)',      'd'),
    (axes[1], cm_norm, 'Confusion Matrix (row %)',       '.2f'),
]:
    im = ax.imshow(data, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(xticks=range(len(present_classes)),
           yticks=range(len(present_classes)),
           xticklabels=present_labels,
           yticklabels=present_labels,
           xlabel='Predicted Class',
           ylabel='True Class',
           title=title)
    thresh = data.max() / 2.0
    for i, j in itertools.product(range(data.shape[0]), range(data.shape[1])):
        val = data[i, j]
        txt = (f'{int(val)}' if fmt == 'd'
               else f'{val:.0%}')
        ax.text(j, i, txt, ha='center', va='center',
                color='white' if val > thresh else 'black', fontsize=11, fontweight='bold')

plt.suptitle(f'Magnitude Class Confusion Matrix — {len(y_true)} test events', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_matrix_v5_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: confusion_matrix')

# ── 4. Per-bin error boxplot ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot of residuals by true class
residuals = y_pred - y_true
class_residuals = [residuals[y_true_cls == c] for c in present_classes]
bp = axes[0].boxplot(class_residuals, labels=present_short,
                     patch_artist=True, notch=False)
colors = ['#4CAF50','#2196F3','#FF9800','#F44336']
for patch, color in zip(bp['boxes'], colors[:len(present_classes)]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set(xlabel='True Magnitude Class', ylabel='Prediction Error (pred−true)',
            title='Error Distribution by Magnitude Class')
axes[0].grid(True, alpha=0.3)

# Scatter: true vs predicted coloured by class
scatter_colors = ['#4CAF50','#2196F3','#FF9800','#F44336']
for i, c in enumerate(present_classes):
    mask = y_true_cls == c
    axes[1].scatter(y_true[mask], y_pred[mask],
                    color=scatter_colors[i], alpha=0.5, s=15, label=present_short[c])
lims = [y_true.min()-0.1, y_true.max()+0.1]
axes[1].plot(lims, lims, 'k--', linewidth=1.5, label='Perfect')
axes[1].plot(lims, [l+0.5 for l in lims], 'r--', linewidth=1, alpha=0.5, label='±0.5')
axes[1].plot(lims, [l-0.5 for l in lims], 'r--', linewidth=1, alpha=0.5)
axes[1].set(xlim=lims, ylim=lims, xlabel='True Magnitude', ylabel='Predicted Magnitude',
            title='Predicted vs True (coloured by class)')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_class_analysis_v5_{TIMESTAMP}.png', dpi=150)
plt.close()
print('Saved: per_class_analysis')

# ── 5. Save extended results ──────────────────────────────────────────────────
extended = {
    'class_accuracy'        : float(class_acc),
    'within_0_5_mag'        : float(within_half),
    'within_0_3_mag'        : float(within_03),
    'within_0_1_mag'        : float(within_01),
    'bias_mean'             : float((y_pred-y_true).mean()),
    'error_std'             : float((y_pred-y_true).std()),
    'class_counts_true'     : {BIN_SHORT[c]: int((y_true_cls==c).sum()) for c in present_classes},
    'class_counts_pred'     : {BIN_SHORT[c]: int((y_pred_cls==c).sum()) for c in present_classes},
}
json.dump(extended, open(f'{RESULTS_DIR}/extended_v5_{TIMESTAMP}.json','w'), indent=2)
print('Saved: extended_results')
print()
print('══════════════════════════════════════════════════════')
print(f'  ✓ Within ±0.5 mag : {within_half*100:.1f}% of test events')
print(f'  ✓ Within ±0.3 mag : {within_03*100:.1f}% of test events')
print('══════════════════════════════════════════════════════')

══════════════════════════════════════════════════════
  FULL VALIDATION SUITE — earthd v5                  
══════════════════════════════════════════════════════

  Regression Metrics
    MAE          : 0.3195  mag units
    RMSE         : 0.4077  mag units
    R²           : -0.2881
    Bias (mean)  : 0.0841  mag units


    Std of error : 0.3990  mag units

  Tolerance Accuracy (% predictions within ±N mag units)
    ±0.5  mag  :  80.6%   (publishable standard)
    ±0.3  mag  :  55.3%   (good)
    ±0.1  mag  :  19.2%   (excellent)

  Classification Accuracy (binned into 4 magnitude classes)
    Class acc  :  50.7%

  Classification Report
              precision    recall  f1-score   support

       Minor       0.61      0.43      0.50      1658
    Moderate       0.44      0.64      0.52      1193
      Strong       0.00      0.00      0.00        51
       Major       0.00      0.00      0.00         3

    accuracy                           0.51      2905
   macro avg       0.26      0.27      0.26      2905
weighted avg       0.53      0.51      0.50      2905

Saved: confusion_matrix


Saved: per_class_analysis
Saved: extended_results

══════════════════════════════════════════════════════
  ✓ Within ±0.5 mag : 80.6% of test events
  ✓ Within ±0.3 mag : 55.3% of test events
══════════════════════════════════════════════════════


## Cell 9 — Save Model & Artifacts

Persists the trained model weights, config, training history, and test results so you can reload and evaluate later without re-training.

In [11]:
# CELL 9 — SAVE MODEL + ARTIFACTS

# Model
model_path   = f'{MODEL_DIR}/model_v5_{TIMESTAMP}.keras'
weights_path = f'{MODEL_DIR}/weights_v5_{TIMESTAMP}.weights.h5'
model.save(model_path)
model.save_weights(weights_path)

# Config snapshot
cfg = {
    'PROCESSED_DIR'       : PROCESSED_DIR,
    'WINDOW_SIZE'         : WINDOW_SIZE,
    'PRE_P_SAMPLES'       : PRE_P_SAMPLES,
    'N_LOC'               : N_LOC,
    'LEARNING_RATE'       : LEARNING_RATE,
    'EPOCHS_REQUESTED'    : EPOCHS,
    'EPOCHS_RAN'          : epochs_ran,
    'BATCH_SIZE'          : BATCH_SIZE,
    'EARLY_STOP_PATIENCE' : EARLY_STOP_PATIENCE,
    'REDUCE_LR_PATIENCE'  : REDUCE_LR_PATIENCE,
    'VAL_FRACTION'        : VAL_FRACTION,
    'MAG_MIN'             : MAG_MIN,
    'MAG_MAX'             : MAG_MAX,
    'TIMESTAMP'           : TIMESTAMP,
}
json.dump(cfg, open(f'{MODEL_DIR}/config_v5_{TIMESTAMP}.json', 'w'), indent=2)

# History
hist_out = {k: [float(v) for v in vs] for k, vs in history.history.items()}
json.dump(hist_out, open(f'{MODEL_DIR}/history_v5_{TIMESTAMP}.json', 'w'), indent=2)

# Results
results = {
    'TIMESTAMP'   : TIMESTAMP,
    'train_events': int(len(y_train)),
    'val_events'  : int(len(y_val)),
    'test_events' : int(len(y_test)),
    'epochs_ran'  : epochs_ran,
    'best_val_mae': float(best_val_mae),
    'test_mae'    : float(mae),
    'test_rmse'   : float(rmse),
    'test_r2'     : float(r2),
    'model_path'  : model_path,
}
json.dump(results, open(f'{RESULTS_DIR}/results_v5_{TIMESTAMP}.json', 'w'), indent=2)

print(f'Model   saved : {model_path}')
print(f'Weights saved : {weights_path}')
print(f'Config  saved : {MODEL_DIR}/config_v5_{TIMESTAMP}.json')
print(f'History saved : {MODEL_DIR}/history_v5_{TIMESTAMP}.json')
print(f'Results saved : {RESULTS_DIR}/results_v5_{TIMESTAMP}.json')


Model   saved : models_v5/model_v5_20260402_105847.keras
Weights saved : models_v5/weights_v5_20260402_105847.weights.h5
Config  saved : models_v5/config_v5_20260402_105847.json
History saved : models_v5/history_v5_20260402_105847.json
Results saved : results_v5/results_v5_20260402_105847.json


## Cell 10 — Summary

Prints a clean end-of-training summary with all key numbers in one place.

In [12]:
# CELL 10 — FINAL SUMMARY

print()
print('══════════════════════════════════════════════════════════')
print('  earthd v5 — COMPLETE TRAINING & EVALUATION SUMMARY     ')
print('══════════════════════════════════════════════════════════')
print()
print('  Dataset (chronological split)')
print(f'    Total events   : {n_total:,}')
print(f'    Train          : {len(y_train):,}  (oldest 80%)')
print(f'    Val            : {len(y_val):,}  (middle 10%)')
print(f'    Test           : {len(y_test):,}  (most recent 10%)')
print(f'    Sample rate    : {TARGET_SR} Hz  ({WINDOW_SIZE} samples / 30s window)')
print()
print('  Training')
print(f'    Epochs ran     : {epochs_ran} / {EPOCHS}')
print(f'    Best val MAE   : {best_val_mae:.4f} mag units')
print()
print('  Regression Metrics (test set)')
print(f'    MAE            : {mae:.4f}  mag units')
print(f'    RMSE           : {rmse:.4f}  mag units')
print(f'    R²             : {r2:.4f}')
print(f'    Bias           : {(y_pred-y_true).mean():.4f}  mag units')
print(f'    Error std      : {(y_pred-y_true).std():.4f}  mag units')
print()
print('  Tolerance Accuracy (test set)')
print(f'    Within ±0.5    : {within_half*100:.1f}%   ← publishable standard')
print(f'    Within ±0.3    : {within_03*100:.1f}%   ← good')
print(f'    Within ±0.1    : {within_01*100:.1f}%   ← excellent')
print()
print('  Classification (4 magnitude bins)')
print(f'    Class accuracy : {class_acc*100:.1f}%')
print()
print('  Saved artifacts')
print(f'    Model          : {model_path}')
print(f'    Best model     : {best_model_path}')
print(f'    Plots          : {RESULTS_DIR}/')
print('══════════════════════════════════════════════════════════')


══════════════════════════════════════════════════════════
  earthd v5 — COMPLETE TRAINING & EVALUATION SUMMARY     
══════════════════════════════════════════════════════════

  Dataset (chronological split)
    Total events   : 29,037
    Train          : 23,229  (oldest 80%)
    Val            : 2,903  (middle 10%)
    Test           : 2,905  (most recent 10%)
    Sample rate    : 20 Hz  (600 samples / 30s window)

  Training
    Epochs ran     : 17 / 100
    Best val MAE   : 0.3427 mag units

  Regression Metrics (test set)
    MAE            : 0.3195  mag units
    RMSE           : 0.4077  mag units
    R²             : -0.2881
    Bias           : 0.0841  mag units
    Error std      : 0.3990  mag units

  Tolerance Accuracy (test set)
    Within ±0.5    : 80.6%   ← publishable standard
    Within ±0.3    : 55.3%   ← good
    Within ±0.1    : 19.2%   ← excellent

  Classification (4 magnitude bins)
    Class accuracy : 50.7%

  Saved artifacts
    Model          : models_v5/mode